#### A Basic Flow of Neural Network when working with 

<p>
Step-0: - Import the necessary Libraries <br>
Step-1: - Set the Device (CUDA, CPU, MLX) <br>
Step-2: - Load the Dataset <br>
Step-3:-  Initialize the basic Network Architecture <br>
Step-4: - Setup the loss function and the optimizer <br>
Step-5: - Write the training loop <br>
Step-6: - Evaluate the results with Accuracy and other metrics <br>



### Step-0:- Importing the Libraries

torch.nn -> All Neural Network Modules (for Nodes), ANN, CNN , loss functions     <br>                           
torch.optim -> Optimization Algos (ADam, SGD)     <br>
torch.nn.functional -> Activation Functions       <br>
torch.utils.data. -> Dataset Management, Minibatches      <br>

torchvision.datasets -> For MNIST dataset     <br>
torchvision.transform -> Augmentation Transform   <br>



In [2]:
import torch
import torch.nn as nn           
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.datasets as datasets
import torchvision.transforms as transforms

### Step-1: - Setup device
**(Also Setting up some additional hyper parameters)**

In [3]:
### Setting up some hyper parameters
input_size = 784        # Since we are testing using MNIST dataset
num_classes = 10
learning_rate = 0.001
batch_size = 64
num_epochs =1

In [4]:
### Another Way to find the current accelerator
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


In [5]:
### This code block is MPS v/s CUDA v/s CPU for device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device("cpu")
device

device(type='mps')

### Step-2: - Load the dataset

In [6]:
from dataset_path import BASE_CODE_DIR_PATH, DATASET_DIR

In [52]:
## Downloading the dataset from actual datasets
train_data = datasets.MNIST(root = DATASET_DIR, train= True, transform=transforms.ToTensor(), download=True)
## Here the transform is to convert the numpy to tensor 
#  Here the root is where we check the root directory if the dataset exists or not, if not we download

In [53]:
### Preparing the dataset and creating dataloader

### THis Dataloader is for data preparation 
## Batch size and shuffle creates the training data, train_data is pytorch Tensor
train_loader = DataLoader(dataset = train_data, batch_size = batch_size, shuffle=True)

In [54]:
# Repeated the same for Test Set
test_data = datasets.MNIST(root= DATASET_DIR, train=False, transform = transforms.ToTensor(), download=True)
test_loader = DataLoader(dataset = test_data, shuffle=True, batch_size=batch_size)

### Step-3: - Intialize the Network

In [55]:
# Any network that we create we need to inherit from nn.Module
class ANN(nn.Module):
    
    # We need to specify the input size (input number of neurons) and output layers
    def __init__(self, input_size:int, num_classes:int):
        super(ANN, self).__init__()         # The first thing, super calls the parent class initialization
        self.fc1= nn.Linear(input_size, 50)
        self.fc2 = nn.Linear(50, num_classes)   # Basic ANN with Linear layer
    
    # Here we created a forward pass 
    def forward(self, x):
        x= F.relu(self.fc1(x))      # Calling RELU for non linearity
        x= self.fc2(x)
        return x

# Basic Initalization of model and a random test to see if architecture is working
# model = ANN(512, 10)
# x= torch.randn(64, 512)       # Here 64 is the mini batch size, 64 samples and 512 matches the input size
# print(model.forward(x).shape)

In [56]:
current_model = ANN(input_size, batch_size).to(device)

### Step-4: - Loss and Optimizer

In [57]:

loss= nn.CrossEntropyLoss()     ## We got the losss function 
optimizer = optim.Adam(current_model.parameters(), lr=learning_rate) ## We set the optimizer, that is pass the learning rate and showed all the model parameter

### Step-5: - Training Loop

In [58]:

## Now we Training the model 

## Step-1: - For loop for num of epochs
for epoch in range(num_epochs):
    
    ## Step-2 : - Then we focus on all the batches that we have in our dataset
    ## Enumerate - Gives us the batch number, the actual data and target
    for batch_idx, (data, targets) in enumerate(train_loader):
        data= data.to(device)
        targets = targets.to(device=device)
        
        ## Since the data was in images, 28*28 image style and model needs 784 flattened we did reshape
        # Doing the right shape
        data= data.reshape(data.shape[0], -1)
        # print(data.shape)
        
        
        ## Step-3: - Do a Forward Pass across the data(one batch) and get the results
        ## Forward pass -> To 
        scores = current_model(data)
        
        ### Step-4: - Calculate the loss on the batch
        loss_val = loss(scores, targets)
        
        ## Step-5: - Now we do a backward pass
        optimizer.zero_grad()       ### This set gradient to 0, for backward pass(makes sure all batches are unique)
        loss_val.backward()
        
        ## Step-6:- Now we Gradient Descent or Adam Step 
        optimizer.step()
        

### Step-6: - Accuracy and Eval

In [59]:
## This is for evaluating the model
def check_accuracy(loader, model):
    if loader.dataset.train:
        print("Checking accuracy on training data")
    else:
        print("Checking accuracy on test data")
    
    ## Num of correct and Number of samples
    num_correct = 0
    num_samples =0
    
    ### This sets the model to evaluation mode -> No updates
    model.eval()
    
    ## No gradient computation set
    with torch.no_grad():
        for x, y in loader:
            ## Getting the input and targets
            x= x.to(device=device)
            y=y.to(device=device)
            x = x.reshape(x.shape[0], -1)
            
            ## Doing the forward pass
            scores =model(x)
            
            ### We get the maximum value and get the predictions (get the index for the right class)
            _, predictions = scores.max(1)
            
            # Check for that batch the number of correct predictions and all predictions(iterate over all batches)
            num_correct +=(predictions==y).sum()
            num_samples +=predictions.size(0)
        
        print(f'Got {num_correct} / {num_samples} with accruacy {float(num_correct)/float(num_samples)*100:.2f}')
    
    # We set the model to training mode again, if we want to retrain 
    model.train()

In [60]:
check_accuracy(train_loader, current_model)

Checking accuracy on training data
Got 55375 / 60000 with accruacy 92.29


In [61]:
check_accuracy(test_loader, current_model)

Checking accuracy on test data
Got 9238 / 10000 with accruacy 92.38
